In [1]:
import os, random, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from imblearn.metrics import specificity_score
from mambapy.vim import VMamba, MambaConfig
from thop import profile

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
class VimEncoder(nn.Module):
    def __init__(self, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        config = MambaConfig(d_model=d_model, n_layers=n_layers, d_state=d_state,
                              bidirectional=True, divide_output=True, pscan=True, use_cuda=False)
        self.encoder = VMamba(config)
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, tokens):
        return self.final_norm(self.encoder(tokens))

In [3]:
class ROIPatchEmbed3D(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32):
        super().__init__()
        self.n_rois = n_rois
        self.patch_size = patch_size
        self.grid_size = roi_size // patch_size
        self.patches_per_roi = self.grid_size ** 3
        self.d_model = d_model
        self.patch_conv = nn.Conv3d(1, d_model, kernel_size=patch_size, stride=patch_size)
        self.roi_embed = nn.Embedding(n_rois, d_model)
        self.depth_embed = nn.Embedding(self.grid_size, d_model)
        self.height_embed = nn.Embedding(self.grid_size, d_model)
        self.width_embed = nn.Embedding(self.grid_size, d_model)
        with torch.no_grad():
            for emb in [self.roi_embed, self.depth_embed, self.height_embed, self.width_embed]:
                emb.weight.mul_(0.02)
        d, h, w = torch.meshgrid(torch.arange(self.grid_size), torch.arange(self.grid_size),
                                  torch.arange(self.grid_size), indexing="ij")
        self.register_buffer("coordinates", torch.stack([d, h, w], dim=-1).reshape(-1, 3), persistent=False)

    def forward(self, rois):
        batch_size, n_rois = rois.shape[:2]
        x = rois.reshape(batch_size * n_rois, 1, rois.shape[-3], rois.shape[-2], rois.shape[-1])
        tokens = self.patch_conv(x).flatten(2).transpose(1, 2)
        tokens = tokens.reshape(batch_size, n_rois, self.patches_per_roi, self.d_model)
        coords = self.coordinates
        spatial = self.depth_embed(coords[:, 0]) + self.height_embed(coords[:, 1]) + self.width_embed(coords[:, 2])
        tokens = tokens + spatial[None, None, :, :] + self.roi_embed.weight[None, :, None, :]
        occupancy = F.max_pool3d((x.abs() > 1e-6).float(), kernel_size=self.patch_size, stride=self.patch_size)
        valid = occupancy.flatten(1).bool().reshape(batch_size, n_rois, self.patches_per_roi)
        tokens = tokens.reshape(batch_size, -1, self.d_model)
        valid = valid.reshape(batch_size, -1)
        tokens = tokens * valid.unsqueeze(-1).to(tokens.dtype)
        return tokens, valid


class VisionMambaBranch(nn.Module):
    """injects pre-resize ROI statistics
    (voxel count, mean/std intensity, bounding box) that were lost
    when each ROI got squashed to a fixed 64^3 during extraction."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        self.n_rois = n_rois
        self.patch_embed = ROIPatchEmbed3D(n_rois, roi_size, patch_size, d_model)
        self.vim = VimEncoder(d_model, n_layers, d_state)
        self.stats_proj = nn.Linear(n_rois * 6, d_model)   # 6 ROIs x 6 stats each -> one d_model vector

    def forward(self, rois, stats):
        tokens, valid = self.patch_embed(rois)
        tokens = self.vim(tokens)

        w = valid.unsqueeze(-1).to(tokens.dtype)
        pooled = (tokens * w).sum(dim=1) / w.sum(dim=1).clamp_min(1.0)

        B = tokens.shape[0]
        tokens_by_roi = tokens.reshape(B, self.n_rois, -1, tokens.shape[-1])
        mask_by_roi = valid.reshape(B, self.n_rois, -1, 1).to(tokens.dtype)
        roi_embeddings = (tokens_by_roi * mask_by_roi).sum(dim=2) / mask_by_roi.sum(dim=2).clamp_min(1.0)

        # fuse the recovered pre-resize stats into the image-derived representation
        # (added, not concatenated -- keeps the classifier input dimension unchanged)
        pooled = pooled + self.stats_proj(stats.reshape(B, -1))
        return pooled, roi_embeddings


class VisionMambaModel(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32, n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, rois, stats, return_roi_embeddings=False):   # note: stats now required
        pooled, roi_emb = self.branch(rois, stats)
        logits = self.classifier(self.dropout(pooled))
        return (logits, roi_emb) if return_roi_embeddings else logits


class MultimodalVisionMambaModel(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32, n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.mri_branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.pet_branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model * 2, n_classes)

    def forward(self, mri_rois, mri_stats, pet_rois, pet_stats, return_roi_embeddings=False):
        mri_pooled, mri_emb = self.mri_branch(mri_rois, mri_stats)
        pet_pooled, pet_emb = self.pet_branch(pet_rois, pet_stats)
        logits = self.classifier(self.dropout(torch.cat([mri_pooled, pet_pooled], dim=1)))
        return (logits, mri_emb, pet_emb) if return_roi_embeddings else logits

In [4]:
COHORT_CSV     = "D:/mamba_model/thesis_cohort_final.csv"
MRI_CACHE_AUG  = "D:/mamba_model/preprocessed_cache_roi64_aug"
PET_CACHE_AUG  = "D:/mamba_model/preprocessed_cache_pet_aug"
CKPT_DIR       = "D:/mamba_model/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

df = pd.read_csv(COHORT_CSV)
if df["subject_id"].nunique() != len(df):
    raise ValueError("Split must be subject-level.")

sessions, labels = df["mri_session"].values, df["outcome_label"].values
X_tv, X_test, y_tv, y_test = train_test_split(sessions, labels, test_size=0.2, random_state=42, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv)
session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

# pre-extracted once (separate notebook, reads raw FastSurfer/PET files from E:/) --
# these two .pkl files contain {subject/session_id: (6, 6) stats array}
mri_stats_dict = pd.read_pickle("D:/mamba_model/mri_roi_stats.pkl")
pet_stats_dict = pd.read_pickle("D:/mamba_model/pet_roi_stats.pkl")

# statistics into how the training features are scaled
train_mri_stats = np.stack([mri_stats_dict[s] for s in X_train])
MRI_MEAN, MRI_STD = train_mri_stats.mean(0, keepdims=True), train_mri_stats.std(0, keepdims=True) + 1e-6

train_pet_stats = np.stack([pet_stats_dict[session_to_subject[s]] for s in X_train])
PET_MEAN, PET_STD = train_pet_stats.mean(0, keepdims=True), train_pet_stats.std(0, keepdims=True) + 1e-6

Train: 126 | Val: 42 | Test: 42


In [5]:
class ROIDataset(Dataset):
    def __init__(self, sessions, labels, cache_dir, is_mri=True, is_train=False):
        self.samples, self.cache_dir, self.is_mri = [], cache_dir, is_mri
        for session_id, label in zip(sessions, labels):
            key = session_id if is_mri else session_to_subject[session_id]
            self.samples.append((session_id, key, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((session_id, key, label, f"aug{seed}"))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        session_id, key, label, version = self.samples[idx]
        rois = np.array(np.load(f"{self.cache_dir}/{key}_{version}.npy", mmap_mode="r"), dtype=np.float32, copy=True)
        # standardise stats using the training-set mean/std computed in Cell 4
        stats = (mri_stats_dict[session_id] - MRI_MEAN[0]) / MRI_STD[0] if self.is_mri else \
                (pet_stats_dict[key] - PET_MEAN[0]) / PET_STD[0]
        return torch.from_numpy(rois).unsqueeze(1), torch.tensor(stats, dtype=torch.float32), torch.tensor(label, dtype=torch.long), key


class MultimodalROIDataset(Dataset):
    def __init__(self, sessions, labels, mri_cache_dir, pet_cache_dir, is_train=False):
        self.samples = []
        self.mri_cache_dir, self.pet_cache_dir = mri_cache_dir, pet_cache_dir
        for session_id, label in zip(sessions, labels):
            subject_id = session_to_subject[session_id]
            self.samples.append((session_id, subject_id, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((session_id, subject_id, label, f"aug{seed}"))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        mri_key, pet_key, label, version = self.samples[idx]
        mri_rois = np.array(np.load(f"{self.mri_cache_dir}/{mri_key}_{version}.npy", mmap_mode="r"), dtype=np.float32, copy=True)
        pet_rois = np.array(np.load(f"{self.pet_cache_dir}/{pet_key}_{version}.npy", mmap_mode="r"), dtype=np.float32, copy=True)
        mri_stats = (mri_stats_dict[mri_key] - MRI_MEAN[0]) / MRI_STD[0]
        pet_stats = (pet_stats_dict[pet_key] - PET_MEAN[0]) / PET_STD[0]
        return (torch.from_numpy(mri_rois).unsqueeze(1), torch.tensor(mri_stats, dtype=torch.float32),
                torch.from_numpy(pet_rois).unsqueeze(1), torch.tensor(pet_stats, dtype=torch.float32),
                torch.tensor(label, dtype=torch.long), mri_key)

In [6]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for rois, stats, labels, _ in loader:
        rois, stats, labels = rois.to(device), stats.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(rois, stats), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, preds_all, labels_all = 0, [], []
    with torch.no_grad():
        for rois, stats, labels, _ in loader:
            rois, stats, labels = rois.to(device), stats.to(device), labels.to(device)
            out = model(rois, stats)
            total_loss += criterion(out, labels).item()
            preds_all.extend(out.argmax(1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    acc = np.mean(np.array(preds_all) == np.array(labels_all))
    tpr = recall_score(labels_all, preds_all, zero_division=0)
    tnr = specificity_score(labels_all, preds_all)
    return total_loss / len(loader), acc, tpr, tnr

def train_epoch_mm(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for mri_rois, mri_stats, pet_rois, pet_stats, labels, _ in loader:
        mri_rois, mri_stats = mri_rois.to(device), mri_stats.to(device)
        pet_rois, pet_stats = pet_rois.to(device), pet_stats.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(mri_rois, mri_stats, pet_rois, pet_stats), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_mm(model, loader, criterion, device):
    model.eval()
    total_loss, preds_all, labels_all = 0, [], []
    with torch.no_grad():
        for mri_rois, mri_stats, pet_rois, pet_stats, labels, _ in loader:
            mri_rois, mri_stats = mri_rois.to(device), mri_stats.to(device)
            pet_rois, pet_stats = pet_rois.to(device), pet_stats.to(device)
            labels = labels.to(device)
            out = model(mri_rois, mri_stats, pet_rois, pet_stats)
            total_loss += criterion(out, labels).item()
            preds_all.extend(out.argmax(1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    acc = np.mean(np.array(preds_all) == np.array(labels_all))
    tpr = recall_score(labels_all, preds_all, zero_division=0)
    tnr = specificity_score(labels_all, preds_all)
    return total_loss / len(loader), acc, tpr, tnr

In [7]:
def measure_inference_time(model, loader, device, is_multimodal, n_batches=20):
    """Amortised per-sample inference time at batch_size=4."""
    model.eval()
    times = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches: break
            if is_multimodal:
                mri_rois, mri_stats, pet_rois, pet_stats, labels, _ = batch  # 6 items now
                mri_rois, mri_stats = mri_rois.to(device), mri_stats.to(device)
                pet_rois, pet_stats = pet_rois.to(device), pet_stats.to(device)
                bs = mri_rois.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time()
                _ = model(mri_rois, mri_stats, pet_rois, pet_stats)
            else:
                rois, stats, labels, _ = batch  # 4 items, not 3 -- this was the bug
                rois, stats = rois.to(device), stats.to(device)
                bs = rois.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time()
                _ = model(rois, stats)
            if device.type == 'cuda': torch.cuda.synchronize()
            times.append((time.time() - t0) / bs)
    return np.mean(times), np.std(times)


def try_compute_flops(model, loader, device, is_multimodal):
    """thop counts standard Conv/Linear layers reliably but may miss
    Vim's custom scan ops -- treat this as an approximate lower bound."""
    try:
        model.eval()
        batch = next(iter(loader))
        with torch.no_grad():
            if is_multimodal:
                mri_rois, mri_stats, pet_rois, pet_stats, labels, _ = batch
                inputs = (mri_rois[:1].to(device), mri_stats[:1].to(device),
                          pet_rois[:1].to(device), pet_stats[:1].to(device))
            else:
                rois, stats, labels, _ = batch  # fixed here too
                inputs = (rois[:1].to(device), stats[:1].to(device))
            macs, _ = profile(model, inputs=inputs, verbose=False)
        return macs * 2
    except Exception as e:
        print(f"  (FLOPs failed: {e})")
        return None


def run_one_seed(seed, model_class, train_loader, val_loader, test_loader, is_multimodal, save_prefix, max_epochs=101, patience=15):
    """Trains one model from a fixed random seed, early-stops on val loss,
    reloads the best checkpoint, then reports test performance + efficiency."""
    torch.manual_seed(seed); torch.cuda.manual_seed(seed); np.random.seed(seed); random.seed(seed)

    model = model_class(d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    train_fn = train_epoch_mm if is_multimodal else train_epoch
    eval_fn = evaluate_mm if is_multimodal else evaluate

    best_val_loss, no_improve, best_epoch, total_time = float("inf"), 0, 0, 0
    save_path = f"{CKPT_DIR}/{save_prefix}_seed{seed}.pt"

    print(f"\n--- Seed {seed} ---")
    print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Val Acc':>8} | {'Val TPR':>8} | {'Val TNR':>8} | {'Time':>6}")
    print("-" * 70)

    for epoch in range(1, max_epochs):
        t0 = time.time()
        train_loss = train_fn(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc, val_tpr, val_tnr = eval_fn(model, val_loader, criterion, device)
        scheduler.step(val_loss)
        epoch_time = time.time() - t0
        total_time += epoch_time
        print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | {val_acc:>8.4f} | {val_tpr:>8.4f} | {val_tnr:>8.4f} | {epoch_time:>5.1f}s")

        if val_loss < best_val_loss:
            best_val_loss, best_epoch, no_improve = val_loss, epoch, 0
            torch.save(model.state_dict(), save_path)
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch}. Best: {best_epoch}")
                break

    model.load_state_dict(torch.load(save_path, weights_only=True))
    test_loss, test_acc, test_tpr, test_tnr = eval_fn(model, test_loader, criterion, device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    inf_mean, inf_std = measure_inference_time(model, test_loader, device, is_multimodal)
    flops = try_compute_flops(model, test_loader, device, is_multimodal)

    print(f"\n  >>> Seed {seed} TEST: Acc={test_acc*100:.1f}% | TPR={test_tpr*100:.1f}% | TNR={test_tnr*100:.1f}% | "
          f"train_time={total_time/60:.1f}min | inf={inf_mean*1000:.2f}ms | {f'{flops/1e9:.2f}GFLOPs' if flops else 'N/A'}")

    return {"seed": seed, "acc": test_acc, "tpr": test_tpr, "tnr": test_tnr, "best_epoch": best_epoch,
            "train_time_sec": total_time, "n_params": n_params, "inf_time_ms": inf_mean * 1000, "flops": flops}

In [8]:
# MRI ONLY
BATCH_SIZE = 4
mri_train_loader = DataLoader(ROIDataset(X_train, y_train, MRI_CACHE_AUG, True, True), batch_size=BATCH_SIZE, shuffle=True)
mri_val_loader   = DataLoader(ROIDataset(X_val, y_val, MRI_CACHE_AUG, True, False), batch_size=BATCH_SIZE, shuffle=False)
mri_test_loader  = DataLoader(ROIDataset(X_test, y_test, MRI_CACHE_AUG, True, False), batch_size=BATCH_SIZE, shuffle=False)

print("=== MRI-ONLY (+ stats): 3-seed run ===")
mri_seed_results = [run_one_seed(s, VisionMambaModel, mri_train_loader, mri_val_loader, mri_test_loader,
                                  False, "vim_stats_mri") for s in [1, 7, 123]]  

=== MRI-ONLY (+ stats): 3-seed run ===

--- Seed 1 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.7461 |     0.7307 |   0.5000 |   0.7619 |   0.2381 |  10.4s
     2 |     0.6887 |     0.7086 |   0.5476 |   0.5238 |   0.5714 |  10.1s
     3 |     0.6732 |     0.6986 |   0.5714 |   0.5714 |   0.5714 |  10.3s
     4 |     0.6579 |     0.6881 |   0.5952 |   0.5714 |   0.6190 |  10.3s
     5 |     0.6475 |     0.6787 |   0.6190 |   0.6190 |   0.6190 |  10.0s
     6 |     0.6230 |     0.6705 |   0.6190 |   0.6190 |   0.6190 |   9.9s
     7 |     0.6307 |     0.6638 |   0.6429 |   0.7619 |   0.5238 |   9.9s
     8 |     0.6102 |     0.6564 |   0.5952 |   0.6667 |   0.5238 |  10.4s
     9 |     0.6084 |     0.6436 |   0.5952 |   0.5238 |   0.6667 |  10.3s
    10 |     0.5923 |     0.6396 |   0.5952 |   0.5238 |   0.6667 |  10.0s
    11 |     0.5943 |     0.6377 |   0.6429 |   0

In [9]:
# PET ONLY
pet_train_loader = DataLoader(ROIDataset(X_train, y_train, PET_CACHE_AUG, False, True), batch_size=BATCH_SIZE, shuffle=True)
pet_val_loader   = DataLoader(ROIDataset(X_val, y_val, PET_CACHE_AUG, False, False), batch_size=BATCH_SIZE, shuffle=False)
pet_test_loader  = DataLoader(ROIDataset(X_test, y_test, PET_CACHE_AUG, False, False), batch_size=BATCH_SIZE, shuffle=False)

print("=== PET-ONLY (+ stats): 3-seed run ===")
pet_seed_results = [run_one_seed(s, VisionMambaModel, pet_train_loader, pet_val_loader, pet_test_loader,
                                  False, "vim_stats_pet") for s in [1, 7, 123]]

=== PET-ONLY (+ stats): 3-seed run ===

--- Seed 1 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.7681 |     0.7488 |   0.4762 |   0.7143 |   0.2381 |  32.0s
     2 |     0.7198 |     0.7295 |   0.4524 |   0.4762 |   0.4286 |  10.7s
     3 |     0.6978 |     0.7234 |   0.5714 |   0.6667 |   0.4762 |  10.4s
     4 |     0.6849 |     0.7158 |   0.5714 |   0.7619 |   0.3810 |  10.0s
     5 |     0.6785 |     0.7069 |   0.5238 |   0.5714 |   0.4762 |   9.9s
     6 |     0.6574 |     0.7014 |   0.5238 |   0.5714 |   0.4762 |  10.0s
     7 |     0.6476 |     0.6990 |   0.6429 |   0.8571 |   0.4286 |  10.0s
     8 |     0.6399 |     0.6910 |   0.6190 |   0.8095 |   0.4286 |  10.4s
     9 |     0.6397 |     0.6761 |   0.5476 |   0.6190 |   0.4762 |  10.1s
    10 |     0.6128 |     0.6738 |   0.5714 |   0.6190 |   0.5238 |  10.0s
    11 |     0.6203 |     0.6729 |   0.6190 |   0

In [10]:
# MIXED MODALITY (MRI & PET)
mm_train_loader = DataLoader(MultimodalROIDataset(X_train, y_train, MRI_CACHE_AUG, PET_CACHE_AUG, True), batch_size=4, shuffle=True)
mm_val_loader   = DataLoader(MultimodalROIDataset(X_val, y_val, MRI_CACHE_AUG, PET_CACHE_AUG, False), batch_size=4, shuffle=False)
mm_test_loader  = DataLoader(MultimodalROIDataset(X_test, y_test, MRI_CACHE_AUG, PET_CACHE_AUG, False), batch_size=4, shuffle=False)

print("=== MIXED MODALITY (+ stats): 3-seed run ===")
mm_seed_results = [run_one_seed(s, MultimodalVisionMambaModel, mm_train_loader, mm_val_loader, mm_test_loader,
                                 True, "vim_stats_mm") for s in [1, 7, 123]]

=== MIXED MODALITY (+ stats): 3-seed run ===

--- Seed 1 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.7348 |     0.7417 |   0.5000 |   0.7619 |   0.2381 |  24.8s
     2 |     0.7148 |     0.7198 |   0.6429 |   0.7143 |   0.5714 |  20.6s
     3 |     0.6668 |     0.6969 |   0.5952 |   0.6190 |   0.5714 |  19.9s
     4 |     0.6371 |     0.6839 |   0.5476 |   0.5238 |   0.5714 |  20.3s
     5 |     0.6505 |     0.6801 |   0.5714 |   0.6667 |   0.4762 |  20.1s
     6 |     0.6298 |     0.6643 |   0.5476 |   0.6190 |   0.4762 |  19.9s
     7 |     0.6261 |     0.6574 |   0.6190 |   0.7619 |   0.4762 |  19.8s
     8 |     0.5984 |     0.6412 |   0.5476 |   0.5714 |   0.5238 |  20.0s
     9 |     0.5979 |     0.6370 |   0.5476 |   0.5238 |   0.5714 |  20.1s
    10 |     0.5914 |     0.6326 |   0.5476 |   0.5238 |   0.5714 |  20.4s
    11 |     0.5807 |     0.6321 |   0.5714

In [11]:
def summarize(results, name):
    accs, tprs, tnrs = [r["acc"] for r in results], [r["tpr"] for r in results], [r["tnr"] for r in results]
    times, infs = [r["train_time_sec"] for r in results], [r["inf_time_ms"] for r in results]
    flops_vals = [r["flops"] for r in results if r["flops"]]
    print(f"\n{name}: Acc={np.mean(accs)*100:.1f}±{np.std(accs,ddof=1)*100:.1f}% | "
          f"TPR={np.mean(tprs)*100:.1f}±{np.std(tprs,ddof=1)*100:.1f}% | "
          f"TNR={np.mean(tnrs)*100:.1f}±{np.std(tnrs,ddof=1)*100:.1f}% | "
          f"Params={results[0]['n_params']:,} | Train={np.mean(times)/60:.1f}m | Inf={np.mean(infs):.2f}ms")

print(f"{'MNA-net (baseline)':<20}: Acc=82.9% | TPR=85.7% | TNR=80.0%")   # Vo et al., single-seed reference
summarize(mri_seed_results, "MRI-only (stats)")
summarize(pet_seed_results, "PET-only (stats)")
summarize(mm_seed_results, "Multimodal (stats)")

MNA-net (baseline)  : Acc=82.9% | TPR=85.7% | TNR=80.0%

MRI-only (stats): Acc=65.1±1.4% | TPR=71.4±0.0% | TNR=58.7±2.7% | Params=46,146 | Train=6.2m | Inf=4.91ms

PET-only (stats): Acc=66.7±2.4% | TPR=66.7±0.0% | TNR=66.7±4.8% | Params=46,146 | Train=7.0m | Inf=6.35ms

Multimodal (stats): Acc=62.7±1.4% | TPR=66.7±0.0% | TNR=58.7±2.7% | Params=92,290 | Train=11.8m | Inf=10.16ms
